# 实验五：单卡 NPU 训练实战

本章在 `1*NPU 910B3 + 16 vCPU + 32GiB` 条件下运行 YOLO 训练。目标是先把 PASCAL VOC 完整跑通，再记录 AMP、batch size、`num_workers` 对吞吐和稳定性的影响。

训练前先确认：数据已经转换、NPU 可见、配置文件路径正确。


## 训练前检查清单

<table style="margin-left: 0; margin-right: auto; text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">检查项</th>
      <th style="text-align: left;">命令</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">NPU 可见</td>
      <td style="text-align: left;"><code>npu-smi info</code></td>
    </tr>
    <tr>
      <td style="text-align: left;">torch-npu 可导入</td>
      <td style="text-align: left;"><code>python -c "import torch, torch_npu; print(torch.npu.device_count())"</code></td>
    </tr>
    <tr>
      <td style="text-align: left;">VOC labels 已转换</td>
      <td style="text-align: left;"><code>ls /mnt/workspace/datasets/voc/labels/train2007 | head</code></td>
    </tr>
    <tr>
      <td style="text-align: left;">train split 已生成</td>
      <td style="text-align: left;"><code>wc -l /mnt/workspace/datasets/voc/splits/train.txt</code></td>
    </tr>
    <tr>
      <td style="text-align: left;">配置路径正确</td>
      <td style="text-align: left;"><code>python -c "import yaml; print(yaml.safe_load(open('src/configs/yolo_ascend.yaml'))['data']['root'])"</code></td>
    </tr>
  </tbody>
</table>


In [ ]:
# ====== 1. 训练前读取配置 ======
from pathlib import Path
import yaml

cfg = yaml.safe_load(Path('src/configs/yolo_ascend.yaml').read_text(encoding='utf-8'))
for key in ['name', 'output_dir']:
    print(f'project.{key}:', cfg['project'][key])
for key in ['root', 'train_list', 'val_list', 'image_size']:
    print(f'data.{key}:', cfg['data'][key])
for key in ['epochs', 'batch_size', 'workers', 'learning_rate', 'warmup_epochs', 'amp']:
    print(f'train.{key}:', cfg['train'][key])


## 启动训练

在终端中执行：

```bash
cd /mnt/workspace/build/cann-learning-hub/contrib/tutorials/Ascend-AI-Lab/03_YOLO_Training_Ascend
bash src/scripts/launch_1npu.sh
```

如果你的仓库路径不同，只要进入 `03_YOLO_Training_Ascend` 目录再运行脚本即可。


In [ ]:
%%bash
cat <<'EOF'
# Choose one launch command after entering the experiment directory.
cd /mnt/workspace/build/cann-learning-hub/contrib/tutorials/Ascend-AI-Lab/03_YOLO_Training_Ascend

# Single-node multi-NPU training. Change NPU_PER_NODE to match your device count.
NPU_PER_NODE=8 bash src/scripts/launch_1node_8p.sh

# Multi-node multi-NPU training. Run the same script on every node with different NODE_RANK.
# Run on rank 0 node:
MASTER_ADDR=<rank0_ip> NNODES=2 NODE_RANK=0 NPU_PER_NODE=8 bash src/scripts/launch_multinode.sh

# Run on rank 1 node:
MASTER_ADDR=<rank0_ip> NNODES=2 NODE_RANK=1 NPU_PER_NODE=8 bash src/scripts/launch_multinode.sh
EOF


## 日志怎么看

训练日志中最重要的是这几类信息：

<table style="margin-left: 0; margin-right: auto; text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">字段</th>
      <th style="text-align: left;">说明</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;"><code>epoch</code></td>
      <td style="text-align: left;">当前训练轮数</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>step</code></td>
      <td style="text-align: left;">当前 epoch 中第几个 batch</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>loss</code></td>
      <td style="text-align: left;">总损失，持续为 NaN 说明训练不稳定或 label 有问题</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>lr</code></td>
      <td style="text-align: left;">当前学习率，warmup 期间会逐步上升</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>imgs/s</code></td>
      <td style="text-align: left;">每秒处理图片数，用于比较吞吐</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>data_time</code></td>
      <td style="text-align: left;">数据读取耗时，偏高时重点调 <code>workers</code> 和数据盘</td>
    </tr>
  </tbody>
</table>

单卡调优时，先看训练是否稳定，再看吞吐是否提升。不要只盯某一个 step 的速度，建议观察多个 epoch 的平均趋势。


In [ ]:
# ====== 2. 生成一张实验记录表模板 ======
rows = [
    {'batch_size': 8, 'workers': 4, 'amp': True, 'imgs_per_sec': '', 'loss_status': '', 'note': ''},
    {'batch_size': 16, 'workers': 8, 'amp': True, 'imgs_per_sec': '', 'loss_status': '', 'note': ''},
    {'batch_size': 24, 'workers': 8, 'amp': True, 'imgs_per_sec': '', 'loss_status': '', 'note': ''},
]

headers = ['batch_size', 'workers', 'amp', 'imgs_per_sec', 'loss_status', 'note']
print('\t'.join(headers))
for row in rows:
    print('\t'.join(str(row[key]) for key in headers))


## 调 batch size

`batch_size` 是每个 step 输入模型的图片数量。增大 batch size 通常能提高 NPU 利用率，但也会增加显存压力。当前 910B3 单卡可以先从 `16` 开始，再根据实际显存和吞吐尝试 `8`、`24` 或 `32`。

修改位置：

```yaml
train:
  batch_size: 16
```

如果出现 OOM，先减小 batch size；如果 loss 变成 NaN，优先减小学习率或增加 warmup。


## 调 DataLoader workers

`num_workers` 是 PyTorch DataLoader 用来读取和解码图片的子进程数量。它不是越大越好：太小会让 NPU 等数据，太大可能让 CPU 抢占严重或内存抖动。

当前 16 vCPU 可以按下面顺序试：

```yaml
train:
  workers: 4
```

```yaml
train:
  workers: 8
```

```yaml
train:
  workers: 12
```

如果 MSPROF 中看到明显 DataLoader gap，就优先调这个参数。


## Checkpoint 与断点恢复

训练输出默认保存在：

```text
runs/yolo_single_npu_voc/
├── last.pt
├── epoch_005.pt
├── epoch_010.pt
└── ...
```

断点恢复时修改配置：

```yaml
train:
  resume: runs/yolo_single_npu_voc/last.pt
```

`checkpoint` 是训练状态快照，里面通常包含模型参数和优化器状态。断点恢复依赖 checkpoint，不是只加载模型权重。


## 本章小结

本章完成单卡训练启动、日志观察、batch size 和 DataLoader workers 的基础调参。下一章用 MSPROF 把“感觉慢”变成可定位的时间线和算子统计。


## 课后练习

请根据本节实验内容完成以下练习。题型包含单选题、多选题、判断题、填空题、简答题和代码设计题。

1. (单选题) 单卡训练实践中，启动训练前最不应该忽略的是？
   - A. 数据路径和环境变量
   - B. 浏览器主题颜色
   - C. PPT 字体
   - D. GitHub 星标数

2. (单选题) 训练日志中的 `loss` 主要用于观察什么？
   - A. 模型优化是否在正常进行
   - B. Git LFS 是否上传成功
   - C. 图片是否压缩
   - D. CANN 是否卸载

3. (单选题) `checkpoint` 的主要作用是？
   - A. 保存训练状态，便于恢复或评估
   - B. 替代数据集
   - C. 删除旧日志
   - D. 提高 XML 解析速度

4. (单选题) 如果训练脚本提示找不到 `splits/train.txt`，最可能的问题是？
   - A. 数据准备没有完成或配置路径不正确
   - B. NPU 温度太低
   - C. 学习率太小
   - D. PPT 没有保存

5. (多选题) 判断一次单卡训练跑通，可以参考哪些输出？
   - A. 训练命令正常退出
   - B. loss 正常打印
   - C. throughput 或 step time 有输出
   - D. checkpoint 文件生成

6. (多选题) 训练速度慢可能来自哪些因素？
   - A. DataLoader workers 太少或 I/O 慢
   - B. batch size 太小导致设备利用率低
   - C. 频繁保存或打印日志
   - D. 数据预处理在 CPU 侧成为瓶颈

7. (多选题) 断点恢复训练时通常需要哪些信息？
   - A. checkpoint 路径
   - B. 模型结构一致
   - C. 优化器/epoch 状态
   - D. 数据配置一致

8. (判断题) 只要生成了 checkpoint，就一定说明训练效果很好。

9. (判断题) 单卡训练实践中，保留关键日志有助于后续性能分析和实验验收。

10. (填空题) 训练脚本中常用 `--batch-size` 控制每次迭代输入的 `____` 数量。

11. (填空题) 断点恢复通常通过参数 `--resume` 指向已有的 `____` 文件。

12. (简答题) 为什么训练实践中要先跑少量 epoch 验证流程？

13. (简答题) 如果 loss 长时间不下降，可以从哪些方面分析？

14. (简答题) 为什么实验报告不建议只写“训练成功”？

15. (代码设计题) 写一条命令，使用单卡脚本训练 2 个 epoch，并显式设置 batch size 和 workers。

> 参考答案见 answer/03.06_single_npu_training_practice_answer.ipynb。
